<a href="https://colab.research.google.com/github/sambslam/PitchXAI-Assignment/blob/main/Task2_Research_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2 - Research Assistant Model (Ollama Modelfile on RunPod)

PitchX assignment, Task 2

This notebook documents a specialized research-assistant model built on Ollama, running on the same RunPod GPU pod used for Task 3 (NVIDIA RTX 6000 Ada, 48 GB VRAM).

How to read this notebook. The model runs on the RunPod pod where Ollama serves it. Google Colab is a separate machine and can't reach the pod, so the cells here document what was run on the pod rather than running live in Colab. The same commands can be run on the provided RunPod access during the interview. Real output from the pod is included below.


## 1. What this task is, and how I read it

The task says to install Ollama and "fine tune it for research." That wording is open to interpretation, so I want to be clear about the choice I made.

There are two ways to read "fine-tune for research." The heavy reading is true weight fine-tuning, where you train the model on a dataset of research examples using a tool like Unsloth, which changes the model's actual weights. The lighter reading is configuring the model to specialize in research through Ollama's Modelfile system, which applies a persistent system prompt and parameters on top of the base model without retraining it.

I did the lighter version. Two reasons. First, the task didn't specify a dataset, and PitchX's product is built around sales and prospect outreach, so "research" most plausibly means researching companies and decision-makers ahead of a sales contact. Building a dataset for that from a guess would be a lot of work resting on an assumption. Second, I chose to put the bulk of my time into Task 1, the voice calling agent, since that's closest to their core product.

I want to be accurate about what this is. A Modelfile with a system prompt is configuration, not weight fine-tuning. The underlying model is unchanged. I can explain how the heavier weight-tuning path would work, and this notebook flags it as the alternative. I went with the approach that produces a working, honest deliverable in the time available.


## 2. Environment

This ran on the same pod as Task 3, so the base setup (Ollama installed, `llama3.1:8b` pulled, models stored on the volume) was already in place.

| Component | Value |
|---|---|
| Platform | RunPod GPU Pod (On-Demand) |
| GPU | NVIDIA RTX 6000 Ada Generation, 48 GB VRAM |
| Driver / CUDA | 550.127.05 / CUDA 12.4 |
| Python | 3.12.3 |
| Model server | Ollama |
| Base model | llama3.1:8b |
| Custom model | sales-researcher |


## 3. What a Modelfile is

A Modelfile is a small text file that defines a custom model in Ollama. It works like a recipe. It doesn't hold the model itself. It points at a base model you already have and layers your changes on top. When you run `ollama create`, Ollama reads the file and produces a new named model that behaves the way you specified.

The point is reuse. Instead of typing a long instruction every time, you bake the behavior into a named model once. After that the model just acts that way whenever you run it.

The parts used here:

`FROM llama3.1:8b` sets the base. The custom model inherits all of Llama 3.1 8B's knowledge and ability. Nothing about the model's weights changes.

`SYSTEM` sets the system prompt. This is a standing instruction that sits in front of every query, so it shapes how the model responds without being repeated. This is the part that specializes the model for research.

`PARAMETER temperature 0.3` lowers randomness so the output stays factual and consistent rather than creative, which suits research.


## 4. The Modelfile

Saved on the pod at `/workspace/Modelfile`.


In [ ]:
FROM llama3.1:8b

SYSTEM """You are a sales research assistant. Your job is to research companies and decision-makers and produce concise, factual summaries that a sales team can act on.

When given a company or person, focus on: what the company does, its industry and size, recent news or signals relevant to outreach, and who the likely decision-makers are. Structure your output clearly. Be factual and do not invent details. If information is missing or uncertain, say so explicitly rather than guessing."""

PARAMETER temperature 0.3

Build the model from the Modelfile and test it:


In [ ]:
# --- pod terminal ---
ollama create sales-researcher -f /workspace/Modelfile

ollama run sales-researcher "Research OpenAI as a sales prospect."

## 5. Output

The model returned a structured research summary without being asked for any particular format. The structure (company summary, what they do, recent signals, decision-makers) came from the system prompt.

```
Company Summary:
- Name: OpenAI
- Industry: Artificial Intelligence, Natural Language Processing
- Size: Private company, several hundred employees
- Location: San Francisco, CA

What they do:
OpenAI develops and commercializes AI technologies, with products including
large language models, image generation, and a developer API.

Recent News and Signals:
- Microsoft investment and partnership
- Expansion of API and product offerings
- Senior hires from other large tech companies

Decision-Makers:
- Sam Altman, CEO
- Greg Brockman, President

Note: roles may change over time.
```

The behavior is right. It researched the prospect and laid the answer out in a form a sales team could use, driven entirely by the system prompt.


## 6. The important caveat

The output above is structured correctly, but the specific facts are not reliable. The model is working from its training data, which is old and in places made up. In the raw run it produced a specific dollar valuation, an exact employee count, a named "Head of Business Development" that looks invented, and it even left a literal `[current date]` placeholder in the text. Those are signs it's generating from memory and a template, not from real information.

This is the main limitation of the approach. A system-prompted model gives you the right research behavior, but it has no access to live data, so it will state outdated or invented facts with confidence. For real sales research the model would need to be connected to live web search or a current data source so it works from real information instead of stale training data.

So the honest way to read this result: it demonstrates the right research behavior and output structure, and the next step to make it actually useful is grounding it in real data.


## 7. The heavier alternative (not done here)

For completeness, the weight fine-tuning path would look like this. Collect or build a dataset of research-style examples (a company or person as input, the kind of summary you want as output). Use a fine-tuning tool such as Unsloth to train Llama 3.1 8B on that dataset, which adjusts the model's actual weights. Convert the result to a format Ollama can run and load it.

That version changes what the model fundamentally does and survives without a system prompt, but it needs a dataset, training time, and more setup. The Modelfile version here is lighter, faster, and reversible, and it was the right fit for the time available. If PitchX confirms they want true weight fine-tuning on a specific dataset, that's the path I'd take.


## 8. Reproduction summary

1. On a pod with Ollama running and `llama3.1:8b` already pulled.
2. Save the Modelfile to `/workspace/Modelfile`.
3. Run `ollama create sales-researcher -f /workspace/Modelfile`.
4. Run `ollama run sales-researcher "Research [company] as a sales prospect."`
5. The model responds in the research-assistant style set by the system prompt.

This notebook is documentation. Live execution is done on RunPod.
